You have been asked to work with a bank to clean the data they collected as part of a recent marketing campaign, which aimed to get customers to take out a personal loan. They plan to conduct more marketing campaigns going forward so would like you to ensure it conforms to the specific structure and data types that they specify so that they can then use the cleaned data you provide to set up a PostgreSQL database, which will store this campaign's data and allow data from future campaigns to be easily imported. 

They have supplied you with a csv file called `"bank_marketing.csv"`, which you will need to clean, reformat, and split the data, saving three final csv files. Specifically, the three files should have the names and contents as outlined below:

## `client.csv`

| column | data type | description | cleaning requirements |
|--------|-----------|-------------|-----------------------|
| `client_id` | `integer` | Client ID | N/A |
| `age` | `integer` | Client's age in years | N/A |
| `job` | `object` | Client's type of job | Change `"."` to `"_"` |
| `marital` | `object` | Client's marital status | N/A |
| `education` | `object` | Client's level of education | Change `"."` to `"_"` and `"unknown"` to `np.NaN` |
| `credit_default` | `bool` | Whether the client's credit is in default | Convert to `boolean` data type:<br> `1` if `"yes"`, otherwise `0` |
| `mortgage` | `bool` | Whether the client has an existing mortgage (housing loan) | Convert to boolean data type:<br> `1` if `"yes"`, otherwise `0` |

<br>

## `campaign.csv`

| column | data type | description | cleaning requirements |
|--------|-----------|-------------|-----------------------|
| `client_id` | `integer` | Client ID | N/A |
| `number_contacts` | `integer` | Number of contact attempts to the client in the current campaign | N/A |
| `contact_duration` | `integer` | Last contact duration in seconds | N/A |
| `previous_campaign_contacts` | `integer` | Number of contact attempts to the client in the previous campaign | N/A |
| `previous_outcome` | `bool` | Outcome of the previous campaign | Convert to boolean data type:<br> `1` if `"success"`, otherwise `0`. |
| `campaign_outcome` | `bool` | Outcome of the current campaign | Convert to boolean data type:<br> `1` if `"yes"`, otherwise `0`. |
| `last_contact_date` | `datetime` | Last date the client was contacted | Create from a combination of `day`, `month`, and a newly created `year` column (which should have a value of `2022`); <br> **Format =** `"YYYY-MM-DD"` |

<br>

## `economics.csv`

| column | data type | description | cleaning requirements |
|--------|-----------|-------------|-----------------------|
| `client_id` | `integer` | Client ID | N/A |
| `cons_price_idx` | `float` | Consumer price index (monthly indicator) | N/A |
| `euribor_three_months` | `float` | Euro Interbank Offered Rate (euribor) three-month rate (daily indicator) | N/A |

## 1 - Split and tidy bank_marketing.csv, storing as three DataFrames called client, campaign, and economics, each containing the columns outlined in the notebook and formatted to the data types listed.

In [2]:
# Pacotes

import pandas as pd
import numpy as np


In [48]:
# Carregando os dados

df = pd.read_csv('./dados/bank_marketing.csv')


In [49]:
# Tratamento de dados 

# - 1 Cliente
df['job'] = df['job'].replace('.','_')
df['education'] = df['education'].str.replace('.','_',regex=False).replace('unknown', np.nan)
df['credit_default'] = df['credit_default'].map({'yes': True}).fillna(False).astype(bool)
df['mortgage'] = df['mortgage'].map({'yes': True}).fillna(False).astype(bool)

# - 2 Campanha
df['previous_outcome'] = df['previous_outcome'].map({'success': True}).fillna(False).astype(bool)
df['campaign_outcome'] = df['campaign_outcome'].map({'yes': True}).fillna(False).astype(bool)


df['month'] = pd.to_datetime(df['month'].str.title(),format='%b').dt.month
df['last_contact_date'] = pd.to_datetime(df['day'].astype(str) + '/' + df['month'].astype(str) + '/' + '2022', format='%d/%m/%Y')



In [44]:
# Criando Bases de Dados

# Cliente

cliente = df[[ 'client_id', 'age', 'job', 'marital', 'education', 'credit_default', 'mortgage']]

# Campanha

campaign = df[['client_id', 'number_contacts', 'contact_duration','previous_campaign_contacts', 'previous_outcome', 'campaign_outcome', 'last_contact_date']]


# Economia

economics = df[['client_id', 'cons_price_idx',  'euribor_three_months']]



In [40]:
# Salvando os arquivos

cliente.to_csv( './dados/cliente.csv', index=False)
campaign.to_csv('./dados/campaign.csv', index=False)
economics.to_csv('./dados/economics.csv', index=False)
